# DSC@UIT 2026 — Task 2 LegalQA — Pipeline extraction-first (Kaggle 2×T4)**Chiến lược.** METEOR của BTC rút gọn được thành$$\text{METEOR}=\frac{R}{0.9+0.1\,r}\times\Big(1-0.5(ch/m)^3\Big),\qquad r=\frac{L_{hyp}}{L_{ref}}$$Ba núm xoay: **R** (recall token — do retrieval), **r** (tỉ lệ độ dài — do calibration), **ch/m** (độ rời rạc — do cách sinh).Hệ quả: *(1)* dài gấp 2 chỉ mất 9% nhưng ngắn một nửa mất ≥47% → **luôn bao gồm thêm khi lưỡng lự**;*(2)* copy nguyên văn liền mạch cho `penalty≈0`, còn paraphrase mất tới 50% → **không bao giờ để model diễn giải lại**.**Model (tổng 2.68B / trần 4B — cả 3 đã được BTC duyệt)**| Vai trò | Model | Params ||---|---|---|| Sparse | BM25 (1-2gram) + char rescorer | 0 || Embedding | `AITeamVN/Vietnamese_Embedding_v2` | 568M || Reranker | `AITeamVN/Vietnamese_Reranker` (ta finetune) | 568M || Generator *(optional)* | `AITeamVN/Vi-Qwen2-1.5B-RAG` | 1.54B || Selector | LightGBM | 0 |**Nguyên tắc thiết kế: retrieve nhỏ, đọc lớn.** Chunk ~450 từ để dense encoding rẻ (512 token), nhưng`assemble()` gộp lại thành **Điều đầy đủ** khi dựng câu trả lời — vì gold trích cả Điều.---## Cách dùng trên Kaggle1. **Settings → Accelerator → GPU T4 ×2**; **Internet → On** (cần tải HF model + `nltk wordnet` + pip).2. Upload dữ liệu thành Kaggle Dataset gồm: `train.json`, `public-official.json`, thư mục `context_*.json`,   và (khuyến nghị) cả thư mục `Scoring-Program-Task-LegalQA/`. Notebook **tự tìm** đường dẫn.3. Chạy tuần tự. Mọi stage **cache** vào `/kaggle/working/artifacts` → cắt session giữa chừng vẫn tiếp tục được.## Kế hoạch session (giới hạn 12h/session)| Session | Stage | Ước lượng | GPU ||---|---|---|---|| **1** | S00–S07 (setup → BM25 → baseline sàn) | ~1.5h | không || **2** | S08–S11 (dense encode → retrieval → rerank → **headroom**) | ~3h | có || **3** | S12–S14 (nhãn METEOR → finetune reranker) | ~5h | có || **4** | S15–S17 (candidates → selector → submission) | ~2h | có || *5* | S18 generator SFT — **chỉ khi headroom cho thấy cần** | ~6h | có |> ⚠️ **Đọc `S11` trước khi làm S12+.** S11 đo ba khoảng trống headroom và nó có quyền phủ quyết thứ tự> finetune. Nếu `recall@50 < 0.85` thì phải finetune **embedding** trước, không phải reranker.

## S00 · ConfigSửa duy nhất ở cell này. `RUN` bật/tắt từng stage; `FORCE` buộc tính lại dù đã có cache.

In [ ]:
import os, sys, io, re, json, time, math, pickle, hashlib, random, unicodedata, gcfrom pathlib import PathSEED = 20260817random.seed(SEED)# ------------------------------------------------------------------ pathsART = Path("/kaggle/working/artifacts")ART.mkdir(parents=True, exist_ok=True)SUB = Path("/kaggle/working/submit"); SUB.mkdir(parents=True, exist_ok=True)FT_DIR = Path("/kaggle/working/reranker_ft")   # định nghĩa ở đây, KHÔNG ở cell S14:                                               # S17 tham chiếu FT_DIR ngay cả khi bỏ qua S14def _autodetect():    """Tìm train.json / public-official.json / thư mục context_*.json.    Override bằng env DSC_DATA_ROOT khi chạy ngoài Kaggle."""    roots = [Path(p) for p in os.environ.get("DSC_DATA_ROOT", "").split(os.pathsep) if p]    roots += [Path("/kaggle/input"), Path("."), Path("..")]    train = pub = ctx = scoring = None    for r in roots:        if not r.exists(): continue        for p in r.rglob("train.json"):            if train is None: train = p        for p in r.rglob("public-official.json"):            if pub is None: pub = p        if ctx is None:            for p in r.rglob("context_*.json"):                ctx = p.parent; break        if scoring is None:            for p in r.rglob("scoring.py"):                scoring = p.parent; break    return train, pub, ctx, scoringP_TRAIN, P_PUBLIC, D_CTX, D_SCORING = _autodetect()print("train  :", P_TRAIN)print("public :", P_PUBLIC)print("ctx dir:", D_CTX)print("scoring:", D_SCORING)assert P_TRAIN and P_PUBLIC and D_CTX, "Không tìm thấy dữ liệu — kiểm tra Kaggle Dataset đã attach chưa."# ------------------------------------------------------------------ modelsM_EMB     = "AITeamVN/Vietnamese_Embedding_v2"M_RERANK  = "AITeamVN/Vietnamese_Reranker"M_GEN     = "AITeamVN/Vi-Qwen2-1.5B-RAG"PARAM_CAP = 4_000_000_000# ------------------------------------------------------------------ hparamsCFG = dict(    # chunking — retrieve nhỏ, đọc lớn    MAX_CHUNK_WORDS   = 450,    MIN_CHUNK_WORDS   = 25,    FALLBACK_WIN      = 400,    FALLBACK_OVERLAP  = 60,    # sparse    BM25_K1 = 1.5, BM25_B = 0.75,    BM25_NGRAM = (1, 2), BM25_MIN_DF = 3, BM25_MAX_FEAT = 1_500_000,    CHAR_NGRAM = (4, 5), CHAR_MAX_FEAT = 400_000,    USE_CHAR_RESCORE = False,   # bật = +~40 phút/8000 query; BM25 1-2gram đã phủ phần lớn lexical    # SPARSE_ONLY=True: bỏ kênh dense -> chạy được HOÀN TOÀN TRÊN CPU, ra submission.zip    # trong ~30 phút. Dùng để lấy mốc leaderboard đầu tiên trước khi có số GPU nào.    # Bỏ qua: S08 (dense), S10 (rerank), S13, S14.    SPARSE_ONLY = False,    # retrieval    TOP_SPARSE = 200, TOP_DENSE = 200, TOP_FUSE = 30, TOP_RERANK = 8,    RRF_K = 60, CIT_BOOST = 3.0,    # dense / rerank    EMB_MAX_LEN = 512, EMB_BATCH = 32,    RERANK_MAX_LEN = 1536, RERANK_BATCH = 8,    # assembly / length    R_GRID = [0.8, 1.0, 1.3, 1.6, 2.0, 2.5, 3.0],    R_DEFAULT = 2.0,    HARD_MAX_WORDS = 1400,    # label generation + finetune    LABEL_N_TRAIN = 5000, LABEL_TOP_K = 30,    FT_GROUP = 8, FT_EPOCHS = 1, FT_LR = 1e-5, FT_TEMP = 0.1, FT_MAXLEN = 512,    # oof    N_FOLDS = 5, OOF_EVAL_N = 1500,)RUN = dict(    S02_scorer=True, S03_data=True, S04_chunk=True, S05_oof=True,    S06_bm25=True, S07_floor=True,    S08_dense=True, S09_retrieve=True, S10_rerank=True, S11_headroom=True,    S12_rsweep=True, S13_labels=True, S14_finetune=True,    S15_candidates=True, S16_selector=True, S17_submit=True,    S18_generator=False,          # bật chỉ khi S11 cho thấy assembly là bottleneck)FORCE = set()                     # ví dụ: FORCE = {"chunks", "emb_chunks"}print("\nartifacts ->", ART)

## S01 · Môi trường, GPU, đếm tham sốĐếm tham số **thật** bằng `sum(p.numel())` — BTC tính cả lớp embedding, và với bge-m3 (vocab 250k)riêng embedding table đã ~256M nên không được ước lượng bằng tên model.

In [ ]:
import subprocessdef pip(*pkgs):    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)try:    import rouge_score  # noqaexcept ImportError:    if D_SCORING and (D_SCORING / "rouge_score").exists():        sys.path.insert(0, str(D_SCORING))          # dùng bản bundled của BTC    else:        pip("rouge-score")for mod, pkg in [("nltk","nltk"), ("sklearn","scikit-learn"), ("scipy","scipy"),                 ("pandas","pandas"), ("pyarrow","pyarrow"), ("lightgbm","lightgbm"),                 ("transformers","transformers"), ("torch", None)]:    try:        __import__(mod)    except ImportError:        if pkg: pip(pkg)import numpy as np, pandas as pd, torch, nltkfor res in ["wordnet", "omw-1.4"]:    try:        nltk.data.find(f"corpora/{res}")    except LookupError:        nltk.download(res, quiet=True)os.environ["TOKENIZERS_PARALLELISM"] = "false"torch.manual_seed(SEED); np.random.seed(SEED)NGPU = torch.cuda.device_count()DEVICES = [f"cuda:{i}" for i in range(NGPU)] or ["cpu"]print("torch", torch.__version__, "| GPUs:", NGPU, DEVICES)for i in range(NGPU):    p = torch.cuda.get_device_properties(i)    print(f"  cuda:{i} {p.name} {p.total_memory/2**30:.1f}GiB sm{p.major}{p.minor}")# T4 = sm75 -> KHÔNG có bf16, phải dùng fp16AMP_DTYPE = torch.float16print("amp dtype:", AMP_DTYPE)print("cpus:", os.cpu_count())def count_params(repo_id, kind="auto"):    """Đếm tham số thật, tải lên CPU rồi giải phóng ngay."""    from transformers import AutoModel, AutoModelForSequenceClassification, AutoModelForCausalLM    cls = {"auto": AutoModel, "seq": AutoModelForSequenceClassification, "lm": AutoModelForCausalLM}[kind]    kw = dict(torch_dtype=torch.float16, trust_remote_code=True)    if kind == "seq": kw["num_labels"] = 1    m = cls.from_pretrained(repo_id, **kw)    n = sum(p.numel() for p in m.parameters())    del m; gc.collect()    return ndef param_budget(include_gen=True):    rows = [("embedding", M_EMB, count_params(M_EMB, "auto")),            ("reranker",  M_RERANK, count_params(M_RERANK, "seq"))]    if include_gen:        rows.append(("generator", M_GEN, count_params(M_GEN, "lm")))    tot = sum(r[2] for r in rows)    print(f"{'role':<11}{'model':<42}{'params':>14}")    for r, m, n in rows: print(f"{r:<11}{m:<42}{n:>14,}")    print(f"{'TOTAL':<53}{tot:>14,}   cap {PARAM_CAP:,}   headroom {PARAM_CAP-tot:,}")    assert tot < PARAM_CAP, "VƯỢT TRẦN 4B — không được nộp cấu hình này"    json.dump({"rows": [[r, m, n] for r, m, n in rows], "total": tot},              open(ART / "param_budget.json", "w"), indent=2)    return tot# Chạy 1 lần rồi comment lại (tốn ~4 phút tải weights):# param_budget(include_gen=RUN["S18_generator"])

## S01b · Cache helperMọi stage đi qua `cache()`. Cắt session bất kỳ lúc nào rồi chạy lại từ đầu → các stage đã xong trả về ngay.

In [ ]:
import numpy as np, pandas as pddef _p(name, fmt): return ART / f"{name}.{fmt}"def _save(obj, path, fmt):    if fmt == "npy":       np.save(path, obj)    elif fmt == "parquet": obj.to_parquet(path, index=False)    elif fmt == "json":    json.dump(obj, open(path, "w", encoding="utf-8"), ensure_ascii=False)    else:                  pickle.dump(obj, open(path, "wb"), protocol=4)def _load(path, fmt):    if fmt == "npy":       return np.load(path, allow_pickle=False)    if fmt == "parquet":   return pd.read_parquet(path)    if fmt == "json":      return json.load(open(path, encoding="utf-8"))    return pickle.load(open(path, "rb"))def cache(name, fn, fmt="pkl"):    path = _p(name, fmt)    if path.exists() and name not in FORCE:        print(f"[cache] {name}.{fmt}")        return _load(path, fmt)    t0 = time.time()    obj = fn()    _save(obj, path, fmt)    print(f"[built] {name}.{fmt}  {time.time()-t0:.1f}s  {path.stat().st_size/2**20:.1f}MiB")    return objclass T:    def __init__(s, m): s.m = m    def __enter__(s): s.t = time.time(); print(f"→ {s.m} ..."); return s    def __exit__(s, *a): print(f"← {s.m}  {time.time()-s.t:.1f}s")

## S02 · Scorer chính thức, bit-for-bitSao y `Scoring-Program-Task-LegalQA/scoring.py`. Ba chi tiết **không được sửa**:`meteor_score([ref.split()], pred.split())` (tách khoảng trắng thuần, dấu câu dính token),`use_stemmer=False`, và tokenizer mặc định của `rouge_score` (loại ký tự tiếng Việt có dấu).ROUGE-L ở đây méo về mặt ngôn ngữ nhưng **phải giữ nguyên** để khớp BTC.

In [ ]:
from nltk.translate.meteor_score import meteor_scorefrom rouge_score import rouge_scorer_ROUGE = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)def meteor_1(ref: str, pred: str) -> float:    return meteor_score([str(ref).split()], str(pred).split())def rougeL_1(ref: str, pred: str) -> float:    return _ROUGE.score(str(ref), str(pred))["rougeL"].fmeasuredef score_official(y_true: dict, y_pred: dict) -> dict:    """y_true: {id: answer_str}; y_pred: {id: {'answer': str}} — đúng shape BTC."""    yp = {k: v["answer"] for k, v in y_pred.items()}    if len(yp) != len(y_true):        raise Exception(f"số key lệch: pred={len(yp)} vs ref={len(y_true)}")    ids = list(yp.keys())    r = np.mean([rougeL_1(y_true[k], yp[k]) for k in ids])    m = np.mean([meteor_1(y_true[k], yp[k]) for k in ids])    return {"rouge": float(r), "meteor": float(m)}if RUN["S02_scorer"]:    s = "Căn cứ khoản 3 Điều 17 Nghị định 90/2017/NĐ-CP quy định như sau: Phạt tiền từ 4.000.000 đồng."    assert meteor_1(s, s) > 0.99, "identical phải ~1.0"    print("identical      :", round(meteor_1(s, s), 4))    print("một nửa        :", round(meteor_1(s, " ".join(s.split()[:len(s.split())//2])), 4))    print("dài gấp đôi    :", round(meteor_1(s, s + " " + s.replace("17", "18")), 4))    print("xáo trộn từ    :", round(meteor_1(s, " ".join(random.sample(s.split(), len(s.split())))), 4))    print("rouge identical:", round(rougeL_1(s, s), 4))

> Đọc kỹ bốn số trên: *"một nửa"* và *"xáo trộn từ"* chính là hai cách mất điểm mà pipeline này được thiết kế> để tránh — ngắn quá, và mất tính liên tục. *"dài gấp đôi"* cho thấy over-length gần như miễn phí.

## S03 · Dữ liệu + duplicate gatePublic QID `129859` trùng chính xác train `88019` → copy thẳng answer. Rẻ, chắc, lấy luôn.

In [ ]:
def norm_q(s: str) -> str:    s = unicodedata.normalize("NFC", str(s)).lower()    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)    return re.sub(r"\s+", " ", s).strip()def load_data():    tr = json.load(open(P_TRAIN, encoding="utf-8"))    pb = json.load(open(P_PUBLIC, encoding="utf-8"))    return tr, pbTRAIN, PUBLIC = load_data()TRAIN_IDS  = sorted(TRAIN.keys())PUBLIC_IDS = sorted(PUBLIC.keys())print(f"train {len(TRAIN)}  public {len(PUBLIC)}")gold_words = np.array([len(TRAIN[i]["answer"].split()) for i in TRAIN_IDS])print(f"gold words: median {np.median(gold_words):.0f}  mean {gold_words.mean():.0f} "      f"p90 {np.percentile(gold_words,90):.0f}  max {gold_words.max()}")# ---- duplicate gate_tr_by_norm = {}for i in TRAIN_IDS:    _tr_by_norm.setdefault(norm_q(TRAIN[i]["question"]), []).append(i)DUP_GATE = {}for i in PUBLIC_IDS:    hit = _tr_by_norm.get(norm_q(PUBLIC[i]["question"]))    if hit:        DUP_GATE[i] = max(hit, key=lambda t: len(TRAIN[t]["answer"]))print("public trùng train (exact sau chuẩn hoá):", DUP_GATE)# nhóm câu trùng trong train -> group id cho OOF (chống leakage)QGROUP = {}for gi, (nq, ids) in enumerate(sorted(_tr_by_norm.items())):    for i in ids: QGROUP[i] = gi# Định nghĩa SỚM: SPARSE_ONLY bỏ qua S08/S09 nên không thể để các map này ở đó.QROW   = {q: k for k, q in enumerate(TRAIN_IDS + PUBLIC_IDS)}TR_POS = {q: k for k, q in enumerate(TRAIN_IDS)}PB_POS = {q: k for k, q in enumerate(PUBLIC_IDS)}print("train: %d câu -> %d nhóm (%d nhóm có >1 câu)" %      (len(TRAIN), len(set(QGROUP.values())),       sum(1 for v in _tr_by_norm.values() if len(v) > 1)))

## S04 · Chunk corpus theo cấu trúc pháp lýƯu tiên ranh giới **Điều**. Ba cạm bẫy được xử lý:1. **Điều xuất hiện hai lần** (mục lục ở đầu + nội dung thật) → giữ bản **dài nhất** theo số Điều.2. **Điều quá dài** → chia tiếp theo **Khoản**, nhưng lưu `parent_key` để `assemble()` gộp lại nguyên Điều.3. **1.125 file thiếu `name`** và **20 passage rỗng** → suy `van_ban` từ header passage / slug link, không crash.`text_raw` giữ **nguyên byte**: không normalize số (`4.000.000`), không sửa dấu câu, không bỏ dòng trắng —vì METEOR tách `.split()` nên mỗi thay đổi định dạng là mất token khớp.

In [ ]:
import globDIEU_RE  = re.compile(r"(?m)^[ \t]*(Điều\s+(\d+[a-zđ]?))\s*[.:\-]?[ \t]*(.*)$")KHOAN_RE = re.compile(r"(?m)^[ \t]*(\d{1,2})[.)]\s")SO_RE    = re.compile(r"(?m)^[ \t]*Số[ \t]*:?[ \t]*([\w][\w/\.\-]*\d[\w/\.\-]*)")VB_TYPES = ["Nghị định","Thông tư liên tịch","Thông tư","Quyết định","Nghị quyết",            "Pháp lệnh","Chỉ thị","Bộ luật","Luật","Công văn","Hướng dẫn","Kế hoạch"]VB_TYPE_RE = re.compile(r"(?i)\b(" + "|".join(VB_TYPES) + r")\b")SLUG_MAP = {"nghi-dinh":"Nghị định","thong-tu":"Thông tư","quyet-dinh":"Quyết định",            "nghi-quyet":"Nghị quyết","phap-lenh":"Pháp lệnh","chi-thi":"Chỉ thị",            "bo-luat":"Bộ luật","luat":"Luật","cong-van":"Công văn"}def extract_van_ban(passage: str, name: str, link: str) -> str:    """-> 'Nghị định 90/2017/NĐ-CP' (best effort, không bao giờ raise)."""    head = (passage or "")[:4000]    mno, mty = SO_RE.search(head), VB_TYPE_RE.search(head)    if mno:        num = mno.group(1).strip(" .,;")        ty = mty.group(1) if mty else None        if ty: return f"{ty[0].upper()+ty[1:].lower()} {num}"        return num    slug = (name or "") or (link or "")    # Mã cơ quan phải khớp CHỮ HOA (case-sensitive). Slug có dạng    #   Thong-tu-17-2022-TT-BGTVT-sua-doi-Thong-tu-12-2020-...    # phần mô tả là chữ thường, nên nếu bật (?i) cho cả nhóm này thì nó ăn luôn    # '-sua-doi-...' rồi .upper() ra 'TT-BGTVT-SUA'. Tách hai bước:    m = re.search(r"(?i)(" + "|".join(SLUG_MAP) + r")-(\d{1,4})-(\d{4})", slug)    if m:        ty, num, yr = SLUG_MAP[m.group(1).lower()], m.group(2), m.group(3)        mc = re.match(r"-([A-Z]{2,6}(?:-[A-Z]{2,10}){0,2})(?=-[a-z0-9]|-?$)", slug[m.end():])        return f"{ty} {num}/{yr}/{mc.group(1)}" if mc else f"{ty} {num}/{yr}"    base = re.sub(r"-\d+$", "", (name or "").replace("-", " ")).strip()    return base[:120] or f"doc {slug[-30:]}"def _win_chunks(text, win, ov):    w = text.split()    if not w: return []    out, i = [], 0    while i < len(w):        out.append(" ".join(w[i:i+win]))        if i + win >= len(w): break        i += win - ov    return outdef _split_exact(text: str, max_w: int):    """Chia theo ranh giới DÒNG, KHÔNG overlap, giữ nguyên từng ký tự.    Bắt buộc dùng bên trong một Điều: `_full_dieu()` ghép các part lại bằng ''.join(),    nên phép chia phải lossless. `_win_chunks` (split/join + overlap) sẽ làm lặp từ    và mất newline -> phá vỡ nguyên tắc copy nguyên byte."""    lines = text.splitlines(keepends=True)    out, cur, n = [], [], 0    for ln in lines:        k = len(ln.split())        if cur and n + k > max_w:            out.append("".join(cur)); cur, n = [], 0        cur.append(ln); n += k    if cur: out.append("".join(cur))    return [o for o in out if o.strip()] or [text]def split_dieu(body: str, max_w: int):    """Chia một Điều dài theo ranh giới Khoản, gom tới ~max_w từ mỗi mảnh."""    marks = [m.start() for m in KHOAN_RE.finditer(body)]    if not marks:        return _split_exact(body, max_w)    segs = [body[:marks[0]]] if marks[0] > 0 else []    for a, b in zip(marks, marks[1:] + [len(body)]):        segs.append(body[a:b])    out, cur, n = [], [], 0    for s in segs:        k = len(s.split())        if cur and n + k > max_w:            out.append("".join(cur)); cur, n = [], 0        cur.append(s); n += k    if cur: out.append("".join(cur))    # HARD CAP: một Khoản đơn lẻ có thể tự nó dài 30k từ (bảng biểu, phụ lục) và lọt qua    # bộ gom ở trên -> phá vỡ giả định 512 token của encoder. Cắt cửa sổ cho phần dư.    final = []    for o in out:        if not o.strip(): continue        if len(o.split()) > 2 * max_w:            final += _split_exact(o, max_w)        else:            final.append(o)    return finaldef build_chunks():    files = sorted(glob.glob(str(D_CTX / "context_*.json")))    rows, empty = [], 0    for fi, f in enumerate(files):        if fi % 1500 == 0: print(f"  {fi}/{len(files)}  rows={len(rows)}")        d = json.load(open(f, encoding="utf-8"))        doc_id, passage = d.get("id"), d.get("passage") or ""        if not passage.strip():            empty += 1; continue        van_ban = extract_van_ban(passage, d.get("name") or "", d.get("link") or "")        ms = list(DIEU_RE.finditer(passage))        # (1) Điều trùng -> giữ span dài nhất        best = {}        for j, m in enumerate(ms):            no = m.group(2)            end = ms[j+1].start() if j+1 < len(ms) else len(passage)            if no not in best or (end - m.start()) > (best[no][1] - best[no][0]):                best[no] = (m.start(), end, m.group(3).strip())        if best:            for no, (a, b, title) in sorted(best.items(), key=lambda kv: kv[1][0]):                body = passage[a:b]                if len(body.split()) < CFG["MIN_CHUNK_WORDS"]: continue                pieces = ([body] if len(body.split()) <= CFG["MAX_CHUNK_WORDS"]                          else split_dieu(body, CFG["MAX_CHUNK_WORDS"]))                for pi, pc in enumerate(pieces):                    rows.append(dict(doc_id=doc_id, van_ban=van_ban, dieu_no=no,                                     dieu_title=title[:300], part=pi, n_parts=len(pieces),                                     parent_key=f"{doc_id}|{no}", text_raw=pc,                                     n_words=len(pc.split()), link=d.get("link") or ""))        else:  # (3) không nhận diện được cấu trúc -> cửa sổ có overlap            for pi, pc in enumerate(_win_chunks(passage, CFG["FALLBACK_WIN"], CFG["FALLBACK_OVERLAP"])):                rows.append(dict(doc_id=doc_id, van_ban=van_ban, dieu_no="", dieu_title="",                                 part=pi, n_parts=-1, parent_key=f"{doc_id}|W{pi}",                                 text_raw=pc, n_words=len(pc.split()), link=d.get("link") or ""))    df = pd.DataFrame(rows)    df["chunk_id"] = np.arange(len(df), dtype=np.int64)    print(f"docs={len(files)} empty={empty} chunks={len(df)}")    return dfif RUN["S04_chunk"]:    CH = cache("chunks", build_chunks, "parquet")    print(CH[["doc_id","van_ban","dieu_no","n_words"]].head(8).to_string())    print("\nn_words:", CH.n_words.describe()[["mean","50%","max"]].to_dict())    print("chunk có dieu_no:", (CH.dieu_no != "").mean().round(3))    print("van_ban parse ok:", (~CH.van_ban.str.startswith("doc ")).mean().round(3))    print("chunk > 2*MAX_CHUNK_WORDS:", int((CH.n_words > 2*CFG["MAX_CHUNK_WORDS"]).sum()))    CH_TEXT = CH.text_raw.tolist()    PARENT_ROWS = CH.groupby("parent_key").indices     # parent_key -> row idx (gộp lại nguyên Điều)    # bảo vệ tính lossless: ghép các part của một Điều phải ra đúng text gốc liền mạch.    # Nếu assert này vỡ thì `_full_dieu()` đang tạo text lặp/mất newline -> sai toàn bộ nhãn METEOR.    _multi = [pk for pk, rs in PARENT_ROWS.items() if len(rs) > 1][:200]    for pk in _multi:        rs = sorted(PARENT_ROWS[pk], key=lambda r: CH.part.values[r])        joined = "".join(CH_TEXT[r] for r in rs)        assert sum(len(CH_TEXT[r]) for r in rs) == len(joined), f"part chồng lấn tại {pk}"    print(f"lossless check: OK trên {len(_multi)} Điều nhiều part")

## S05 · OOF split (group theo câu hỏi chuẩn hoá)Train có nhóm câu hỏi trùng nhau; nếu để chúng rơi khác fold thì OOF sẽ lạc quan giả và mọi quyết địnhsau đó đều lệch. `GroupKFold` trên `QGROUP`.

In [ ]:
from sklearn.model_selection import GroupKFolddef build_oof():    ids = np.array(TRAIN_IDS)    grp = np.array([QGROUP[i] for i in ids])    fold = np.full(len(ids), -1, dtype=np.int8)    for k, (_, te) in enumerate(GroupKFold(n_splits=CFG["N_FOLDS"]).split(ids, groups=grp)):        fold[te] = k    return pd.DataFrame({"qid": ids, "group": grp, "fold": fold})if RUN["S05_oof"]:    OOF = cache("oof_split", build_oof, "parquet")    print(OOF.fold.value_counts().sort_index().to_dict())    rng = np.random.default_rng(SEED)    EVAL_IDS = list(rng.choice(OOF.qid.values, size=min(CFG["OOF_EVAL_N"], len(OOF)), replace=False))    print("EVAL_IDS:", len(EVAL_IDS))

## S06 · Sparse: BM25 (1-2gram) + citation parserBM25 tự cài bằng `scipy` sparse — nhanh hơn `rank_bm25` vài bậc trên 8.000 query × ~125k chunk,và không thêm dependency. Kênh char `4-5gram` **không** làm index toàn corpus (quá tốn RAM) mà chỉ**rescore trên tập ứng viên** — bắt lỗi chính tả và biến thể viết số văn bản.Citation parser tách hai pass độc lập (số Điều / mã văn bản / số khoản) thay vì một regex tổ hợp — dễ vỡ hơn nhiều.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizerimport scipy.sparse as spdef bm25_fit(texts, ngram, min_df, max_feat, k1, b):    cv = CountVectorizer(ngram_range=ngram, min_df=min_df, max_features=max_feat,                         lowercase=True, dtype=np.int32)    X = cv.fit_transform(texts)                       # docs x terms    N = X.shape[0]    df = np.bincount(X.indices, minlength=X.shape[1])    idf = np.log(1.0 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)    dl = np.asarray(X.sum(1)).ravel().astype(np.float32)    avg = float(dl.mean())    Xc = X.tocsr().astype(np.float32)    rows = np.repeat(np.arange(N), np.diff(Xc.indptr))    denom = Xc.data + k1 * (1 - b + b * dl[rows] / avg)    Xc.data = Xc.data * (k1 + 1) / denom              # doc-side BM25 weight    return dict(cv=cv, W=Xc, idf=idf)def bm25_query(idx, queries, topk):    q = idx["cv"].transform(queries).astype(np.float32)      # nq x terms    q = q.multiply(idx["idf"][None, :]).tocsr()    S = (q @ idx["W"].T).toarray()                           # nq x ndocs    top = np.argpartition(-S, topk, axis=1)[:, :topk]    ord_ = np.take_along_axis(S, top, 1).argsort(1)[:, ::-1]    top = np.take_along_axis(top, ord_, 1)    return top, np.take_along_axis(S, top, 1)# ------------------------------------------------------------ citationVB_NUM_RE = re.compile(r"(?i)\b(" + "|".join(VB_TYPES) + r")\s+(?:số\s+)?(\d{1,4}\s*/\s*\d{4}(?:\s*/\s*[A-ZĐa-zđ\-]+)?)")DIEU_NUM_RE = re.compile(r"(?i)\bĐiều\s+(\d+[a-zđ]?)")KHOAN_NUM_RE = re.compile(r"(?i)\bkhoản\s+(\d+)")def parse_cit(text: str) -> dict:    t = str(text)    vbs = [a[0].upper() + a[1:].lower() + " " + re.sub(r"\s+", "", b)           for a, b in VB_NUM_RE.findall(t)]    return dict(van_ban=list(dict.fromkeys(vbs)),                dieu=list(dict.fromkeys(DIEU_NUM_RE.findall(t))),                khoan=list(dict.fromkeys(KHOAN_NUM_RE.findall(t))))if RUN["S06_bm25"]:    with T("BM25 fit"):        BM = cache("bm25", lambda: bm25_fit(CH_TEXT, CFG["BM25_NGRAM"], CFG["BM25_MIN_DF"],                                            CFG["BM25_MAX_FEAT"], CFG["BM25_K1"], CFG["BM25_B"]))    print("vocab", len(BM["cv"].vocabulary_), "| W nnz", BM["W"].nnz)    # index tra cứu citation -> chunk row    VB_DIEU = {}    for r, (vb, dn) in enumerate(zip(CH.van_ban.values, CH.dieu_no.values)):        if dn: VB_DIEU.setdefault((vb, str(dn)), []).append(r)    print("cặp (văn bản, Điều) duy nhất:", len(VB_DIEU))    q = TRAIN[TRAIN_IDS[0]]["question"]    print("\nví dụ citation trong question:", parse_cit(q))    print("ví dụ citation trong answer  :", parse_cit(TRAIN[TRAIN_IDS[0]]["answer"])["van_ban"][:3])    n_cit = sum(1 for i in TRAIN_IDS if parse_cit(TRAIN[i]["answer"])["van_ban"])    print(f"answer có mã văn bản: {n_cit}/{len(TRAIN_IDS)}")

## S07 · `assemble()` — **nguồn chân lý duy nhất**Mọi thứ downstream phụ thuộc cell này: nhãn METEOR ở S13, finetune ở S14, candidate ở S15, submission ở S17.Sửa `assemble()` mà không tính lại nhãn = **train/serve mismatch**, và reranker sẽ tối ưu cho một hàm không tồn tại.Bốn quy tắc:- Mở đầu `"Căn cứ ..."` (57% gold dùng mẫu này) — cho ~15–25 token khớp **liền mạch** mỗi câu, gần như miễn phí.- Body **copy nguyên byte**, gộp sub-chunk về **Điều đầy đủ** (`parent_key`).- Thiếu so với `L*` → **mở rộng** sang chunk kế tiếp (bất đối xứng 5:1 nghiêng về over-length).- Vượt `L*` → cắt **theo ranh giới dòng**, không cắt giữa dòng, để giữ `ch/m` thấp.Không sinh phần `"Trước đây, căn cứ ..."`: chỉ ≤39/7000 gold có, thêm vào là đẩy `r` vô ích.

In [ ]:
QTYPE_RULES = [    ("xu_phat",   r"xử phạt|phạt tiền|mức phạt|bị phạt|vi phạm hành chính"),    ("mau_bieu",  r"^\s*mẫu\b|biểu mẫu|tờ khai|mẫu số"),    ("ho_so",     r"hồ sơ|thủ tục|trình tự|các bước|đăng ký|cấp giấy"),    ("trach_nhiem", r"trách nhiệm|nhiệm vụ|quyền hạn|nghĩa vụ|có quyền"),    ("dieu_kien", r"điều kiện|tiêu chuẩn|yêu cầu gì|tiêu chí"),    ("dinh_nghia", r"là gì|được hiểu|khái niệm|định nghĩa"),    ("muc_huong", r"mức hưởng|trợ cấp|phụ cấp|tiền lương|bao nhiêu tiền|hệ số"),    ("thoi_han",  r"thời hạn|thời gian|bao lâu|khi nào|mấy ngày"),    ("tham_quyen", r"thẩm quyền|cơ quan nào|ai có quyền|do ai"),]def qtype(q: str) -> str:    ql = str(q).lower()    for name, pat in QTYPE_RULES:        if re.search(pat, ql): return name    return "khac"def build_len_table():    d = {}    for i in TRAIN_IDS:        d.setdefault(qtype(TRAIN[i]["question"]), []).append(len(TRAIN[i]["answer"].split()))    tab = {k: float(np.median(v)) for k, v in d.items()}    tab["__global__"] = float(np.median(gold_words))    return tabif RUN["S07_floor"]:    LEN_TAB = cache("len_table", build_len_table, "json")    cnt = {}    for i in TRAIN_IDS: cnt[qtype(TRAIN[i]["question"])] = cnt.get(qtype(TRAIN[i]["question"]), 0) + 1    print(f"{'qtype':<14}{'n':>6}{'median gold words':>20}")    for k in sorted(cnt, key=lambda x: -cnt[x]):        print(f"{k:<14}{cnt[k]:>6}{LEN_TAB[k]:>20.0f}")def target_words(question: str, r: float) -> int:    base = LEN_TAB.get(qtype(question), LEN_TAB["__global__"])    return int(min(CFG["HARD_MAX_WORDS"], max(60, round(r * base))))def _full_dieu(row_idx: int) -> str:    """Gộp mọi sub-chunk cùng parent_key -> Điều đầy đủ, đúng thứ tự part."""    pk = CH.parent_key.values[row_idx]    rows = PARENT_ROWS.get(pk, np.array([row_idx]))    if len(rows) == 1: return CH_TEXT[rows[0]]    rows = sorted(rows, key=lambda r: CH.part.values[r])    return "".join(CH_TEXT[r] for r in rows)def _head(row_idx: int) -> str:    vb = CH.van_ban.values[row_idx]; dn = CH.dieu_no.values[row_idx]    ti = CH.dieu_title.values[row_idx]    if dn:        s = f"Căn cứ Điều {dn} {vb}"        return s + (f" quy định về {ti.rstrip('.')} như sau:" if ti else " quy định như sau:")    return f"Căn cứ {vb} quy định như sau:"def _trim(text: str, target: int) -> str:    if len(text.split()) <= target: return text    lines, out, n = text.split("\n"), [], 0    for ln in lines:        k = len(ln.split())        if out and n + k > target: break        out.append(ln); n += k    return "\n".join(out) if out else " ".join(text.split()[:target])def assemble(row_idxs, question: str, r: float = None, expand: bool = True) -> str:    """CANONICAL. row_idxs: chunk row đã xếp theo độ tin cậy giảm dần."""    r = CFG["R_DEFAULT"] if r is None else r    tgt = target_words(question, r)    parts, seen, n = [], set(), 0    for ri in row_idxs:        pk = CH.parent_key.values[ri]        if pk in seen: continue        seen.add(pk)        body = _full_dieu(ri) if expand else CH_TEXT[ri]        blk = _head(ri) + "\n" + body.strip()        parts.append(blk); n += len(blk.split())        if n >= tgt: break    if not parts: return ""    txt = "\n".join(parts)    return _trim(txt, tgt) if n > tgt * 1.25 else txt

## S07b · Ba mốc sàn (không cần GPU)Đo trước khi làm bất cứ thứ gì có GPU. Ba con số này định hình mọi quyết định còn lại:| Mốc | Ý nghĩa ||---|---|| **scaffold** | chỉ template + boilerplate pháp lý, **không** retrieval → sàn tuyệt đối || **kNN answer-bank** | trả nguyên answer của train question gần nhất → sàn không cần corpus || **BM25 top-1** | `assemble()` trên chunk BM25 top-1 → sàn của nhánh extractive |Nếu BM25 top-1 chưa vượt rõ kNN thì lỗi ở chunking/assembly, **không** phải ở model — và thêm GPU sẽ không cứu.

In [ ]:
SCAFFOLD = ("Căn cứ quy định của pháp luật hiện hành, đối với trường hợp này được quy định như sau:\n"            "1. Về nguyên tắc chung, cơ quan, tổ chức, cá nhân có liên quan phải tuân thủ các quy định "            "tại Điều, khoản của văn bản quy phạm pháp luật có hiệu lực.\n"            "2. Về trách nhiệm, các bên có trách nhiệm thực hiện đầy đủ nghĩa vụ theo quy định.\n"            "- Trường hợp không thực hiện đúng quy định thì tuỳ theo tính chất, mức độ vi phạm sẽ bị "            "xử lý theo quy định của pháp luật.\n"            "- Đối với các nội dung khác không quy định tại văn bản này thì thực hiện theo quy định "            "của pháp luật có liên quan.\n"            "Theo đó, việc áp dụng phải căn cứ vào quy định cụ thể nêu trên.")def eval_preds(pred: dict, ids) -> dict:    m = np.array([meteor_1(TRAIN[i]["answer"], pred[i]) for i in ids])    r = np.array([rougeL_1(TRAIN[i]["answer"], pred[i]) for i in ids])    return dict(meteor=float(m.mean()), rouge=float(r.mean()),                meteor_p10=float(np.percentile(m, 10)), meteor_p90=float(np.percentile(m, 90)))if RUN["S07_floor"]:    ids = EVAL_IDS[:600]    # ---- floor 1: scaffold, quét theo r    print("floor 1 · scaffold")    for r in [1.0, 1.5, 2.0, 3.0]:        pred = {}        for i in ids:            tgt = target_words(TRAIN[i]["question"], r)            reps = max(1, math.ceil(tgt / len(SCAFFOLD.split())))            pred[i] = _trim("\n".join([SCAFFOLD] * reps), tgt)        print("   r=%.1f  %s" % (r, eval_preds(pred, ids)))    # ---- floor 2: kNN answer-bank    print("floor 2 · kNN answer-bank")    pool = [i for i in TRAIN_IDS if i not in set(ids)]    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2)    Xp = vec.fit_transform([TRAIN[i]["question"] for i in pool])    Xq = vec.transform([TRAIN[i]["question"] for i in ids])    from sklearn.preprocessing import normalize    S = (normalize(Xq) @ normalize(Xp).T).toarray()    nn = S.argmax(1)    pred = {i: TRAIN[pool[nn[k]]]["answer"] for k, i in enumerate(ids)}    print("   %s" % eval_preds(pred, ids))    # ---- floor 3: BM25 top-1 + assemble    print("floor 3 · BM25 top-1 + assemble")    top, _ = bm25_query(BM, [TRAIN[i]["question"] for i in ids], 5)    for r in [1.0, 2.0, 2.5]:        pred = {i: assemble([top[k, 0]], TRAIN[i]["question"], r) for k, i in enumerate(ids)}        print("   r=%.1f  %s" % (r, eval_preds(pred, ids)))    print("floor 3b · BM25 top-3 + assemble (r=2.0)")    pred = {i: assemble(list(top[k, :3]), TRAIN[i]["question"], 2.0) for k, i in enumerate(ids)}    print("   %s" % eval_preds(pred, ids))

## S08 · Dense encoding trên 2 GPUHai thread, mỗi thread một bản model trên một device. GIL được nhả trong lúc CUDA chạy nênpattern này đạt gần 2× mà không cần DDP/multiprocessing (rất dễ vỡ trong notebook).Chunk sort theo độ dài trước khi batch → giảm padding, nhanh thêm ~30%.`EMB_MAX_LEN=512` cho kênh recall; reranker mới dùng 1536 (chính khả năng 8k của bge-m3 cho phép điều đó —các model PhoBERT-based chặn ở 258 token nên không dùng được cho việc này).

In [ ]:
import torch.nn.functional as Ffrom concurrent.futures import ThreadPoolExecutorfrom transformers import AutoTokenizer, AutoModeldef _encode_shard(device, texts, max_len, batch, is_query=False):    tok = AutoTokenizer.from_pretrained(M_EMB)    mdl = AutoModel.from_pretrained(M_EMB, torch_dtype=AMP_DTYPE).to(device).eval()    order = np.argsort([len(t) for t in texts])           # sort by length    out = np.zeros((len(texts), mdl.config.hidden_size), dtype=np.float16)    with torch.inference_mode():        for s in range(0, len(order), batch):            idx = order[s:s+batch]            enc = tok([texts[i] for i in idx], padding=True, truncation=True,                      max_length=max_len, return_tensors="pt").to(device)            h = mdl(**enc).last_hidden_state[:, 0]         # bge-m3 dense = CLS            out[idx] = F.normalize(h, dim=-1).cpu().numpy().astype(np.float16)            if s % (batch * 200) == 0:                print(f"   [{device}] {s}/{len(order)}", flush=True)    del mdl; torch.cuda.empty_cache()    return outdef encode(texts, max_len, batch):    if len(DEVICES) == 1:        return _encode_shard(DEVICES[0], texts, max_len, batch)    cuts = np.linspace(0, len(texts), len(DEVICES) + 1).astype(int)    shards = [(DEVICES[d], texts[cuts[d]:cuts[d+1]]) for d in range(len(DEVICES))]    with ThreadPoolExecutor(max_workers=len(DEVICES)) as ex:        res = list(ex.map(lambda a: _encode_shard(a[0], a[1], max_len, batch), shards))    return np.concatenate(res, 0)if RUN["S08_dense"]:    with T(f"encode {len(CH_TEXT)} chunks"):        EMB_C = cache("emb_chunks", lambda: encode(CH_TEXT, CFG["EMB_MAX_LEN"], CFG["EMB_BATCH"]), "npy")    with T("encode queries"):        ALLQ = [TRAIN[i]["question"] for i in TRAIN_IDS] + [PUBLIC[i]["question"] for i in PUBLIC_IDS]        EMB_Q = cache("emb_queries", lambda: encode(ALLQ, 128, 128), "npy")    print("emb_chunks", EMB_C.shape, "| emb_queries", EMB_Q.shape)

## S09 · Retrieval: BM25 + dense + char rescorer → RRF + citation boostRRF (`1/(60+rank)`) thay vì cộng điểm thô — bất biến với thang điểm của từng kênh, không cần tune trọng số.Citation boost là **soft** (nhân điểm), không hard-filter: đề phòng parser bắt sai mã văn bản.

In [ ]:
_EMB_C32 = Nonedef _chunk_mat():    """float32 view của EMB_C, tạo MỘT lần. Không được .astype() trong vòng lặp:    125k x 1024 float32 = 512MiB mỗi lần gọi."""    global _EMB_C32    if _EMB_C32 is None:        _EMB_C32 = np.ascontiguousarray(EMB_C, dtype=np.float32)        print(f"   [dense] chunk matrix {_EMB_C32.nbytes/2**20:.0f}MiB cached")    return _EMB_C32def dense_query(qrows, topk):    Q = _chunk_mat()    out_i = np.zeros((len(qrows), topk), dtype=np.int64)    out_s = np.zeros((len(qrows), topk), dtype=np.float32)    B = 128    for s in range(0, len(qrows), B):        qv = EMB_Q[qrows[s:s+B]].astype(np.float32)        S = qv @ Q.T        t = np.argpartition(-S, topk, axis=1)[:, :topk]        o = np.take_along_axis(S, t, 1).argsort(1)[:, ::-1]        out_i[s:s+B] = np.take_along_axis(t, o, 1)        out_s[s:s+B] = np.take_along_axis(np.take_along_axis(S, t, 1), o, 1)    return out_i, out_s_CHAR_VEC = Nonedef char_rescore(question, cand_rows):    global _CHAR_VEC    if _CHAR_VEC is None:        _CHAR_VEC = TfidfVectorizer(analyzer="char_wb", ngram_range=CFG["CHAR_NGRAM"],                                    max_features=CFG["CHAR_MAX_FEAT"], sublinear_tf=True)        _CHAR_VEC.fit([CH_TEXT[i][:2000] for i in                       np.random.default_rng(SEED).choice(len(CH_TEXT), 20000, replace=False)])    from sklearn.preprocessing import normalize    A = normalize(_CHAR_VEC.transform([question]))    B = normalize(_CHAR_VEC.transform([CH_TEXT[r][:2000] for r in cand_rows]))    return (A @ B.T).toarray().ravel()def rrf(rank_lists, k):    sc = {}    for lst in rank_lists:        for rank, row in enumerate(lst):            sc[row] = sc.get(row, 0.0) + 1.0 / (k + rank + 1)    return scdef retrieve(questions, qrows, topk=None):    topk = topk or CFG["TOP_FUSE"]    sp_i, _ = bm25_query(BM, questions, CFG["TOP_SPARSE"])    dn_i = None    if not CFG["SPARSE_ONLY"]:        dn_i, _ = dense_query(np.asarray(qrows), CFG["TOP_DENSE"])    out = np.zeros((len(questions), topk), dtype=np.int64)    for n, q in enumerate(questions):        lists = [list(sp_i[n])]        if dn_i is not None: lists.append(list(dn_i[n]))        if CFG["USE_CHAR_RESCORE"]:            cands = list(dict.fromkeys([x for l in lists for x in l]))[:100]            cs = char_rescore(q, cands)            lists.append([cands[i] for i in np.argsort(-cs)])        sc = rrf(lists, CFG["RRF_K"])        cit = parse_cit(q)        if cit["van_ban"] and cit["dieu"]:            for vb in cit["van_ban"]:                for dn in cit["dieu"]:                    for r in VB_DIEU.get((vb, dn), []):                        sc[r] = sc.get(r, 0.0) * CFG["CIT_BOOST"] + 1.0        best = sorted(sc.items(), key=lambda kv: -kv[1])[:topk]        rows = [b[0] for b in best]        out[n, :len(rows)] = rows        if len(rows) < topk: out[n, len(rows):] = rows[-1] if rows else 0    return outif RUN["S09_retrieve"]:    def _cands_train():        qs = [TRAIN[i]["question"] for i in TRAIN_IDS]        qr = [QROW[i] for i in TRAIN_IDS]        chunks_out = []        B = 500        for s in range(0, len(qs), B):            chunks_out.append(retrieve(qs[s:s+B], qr[s:s+B]))            print(f"   retrieve train {s+B}/{len(qs)}", flush=True)        return np.concatenate(chunks_out, 0)    CAND_TR = cache("cand_train", _cands_train, "npy")    def _cands_pub():        qs = [PUBLIC[i]["question"] for i in PUBLIC_IDS]        qr = [QROW[i] for i in PUBLIC_IDS]        return retrieve(qs, qr)    CAND_PB = cache("cand_public", _cands_pub, "npy")    print("cand_train", CAND_TR.shape, "| cand_public", CAND_PB.shape)

## S10 · Reranker (cross-encoder) trên 2 GPU

In [ ]:
from transformers import AutoModelForSequenceClassificationdef _rerank_shard(device, pairs, ckpt, max_len, batch):    tok = AutoTokenizer.from_pretrained(ckpt)    mdl = AutoModelForSequenceClassification.from_pretrained(        ckpt, num_labels=1, torch_dtype=AMP_DTYPE).to(device).eval()    out = np.zeros(len(pairs), dtype=np.float32)    order = np.argsort([len(b) for _, b in pairs])    with torch.inference_mode():        for s in range(0, len(order), batch):            idx = order[s:s+batch]            a = [pairs[i][0] for i in idx]; b = [pairs[i][1] for i in idx]            enc = tok(a, b, padding=True, truncation=True, max_length=max_len,                      return_tensors="pt").to(device)            out[idx] = mdl(**enc).logits[:, 0].float().cpu().numpy()            if s % (batch * 400) == 0: print(f"   [{device}] {s}/{len(order)}", flush=True)    del mdl; torch.cuda.empty_cache()    return outdef rerank(pairs, ckpt=None, max_len=None, batch=None):    ckpt = ckpt or M_RERANK    max_len = max_len or CFG["RERANK_MAX_LEN"]; batch = batch or CFG["RERANK_BATCH"]    if len(DEVICES) == 1:        return _rerank_shard(DEVICES[0], pairs, ckpt, max_len, batch)    cuts = np.linspace(0, len(pairs), len(DEVICES) + 1).astype(int)    with ThreadPoolExecutor(max_workers=len(DEVICES)) as ex:        res = list(ex.map(lambda d: _rerank_shard(DEVICES[d], pairs[cuts[d]:cuts[d+1]],                                                  ckpt, max_len, batch), range(len(DEVICES))))    return np.concatenate(res, 0)def rerank_set(qids, src, cand, topn, ckpt=None):    """-> (n, TOP_FUSE) điểm reranker cho từng ứng viên."""    pairs, meta = [], []    for qi in qids:        k = (TR_POS if src is TRAIN else PB_POS)[qi]        for r in cand[k, :topn]:            pairs.append((src[qi]["question"], CH_TEXT[r])); meta.append((qi, int(r)))    sc = rerank(pairs, ckpt=ckpt)    d = {}    for (qi, r), s in zip(meta, sc): d.setdefault(qi, {})[r] = float(s)    return dif RUN["S10_rerank"]:    ids = EVAL_IDS[:600]    with T(f"rerank {len(ids)}x{CFG['TOP_FUSE']} pairs (zero-shot)"):        RR0 = cache("rr_zeroshot_eval", lambda: rerank_set(ids, TRAIN, CAND_TR, CFG["TOP_FUSE"]))    print("ví dụ điểm:", sorted(list(RR0[ids[0]].values()), reverse=True)[:5])

## S11 · ⚠️ HEADROOM — cell quyết định thứ tự finetuneBa khoảng trống, cộng lại bằng đúng phần điểm đang mất:| Khoảng | Thuộc về | Finetune gì ||---|---|---|| `1.0 − oracle@200` | template / assembly | LLM, rule S07 || `oracle@200 − oracle@50` | recall | **embedding** || `oracle@50 − reranked@1` | precision | **reranker** |**Khoảng nào lớn nhất thì làm cái đó trước.** Prior của tôi là reranker, nhưng cell này có quyền phủ quyết.Cụ thể: `recall-proxy` thấp (oracle@200 ≫ oracle@50) ⇒ đổi sang embedding-first.

In [ ]:
def oracle_at(qids, cand, pos_map, k, r=None):    """max METEOR đạt được nếu chọn đúng chunk tốt nhất trong top-k.    pos_map: qid -> hàng trong `cand` (tách khỏi TR_POS để dùng được với mọi bảng ứng viên)."""    best, arg = {}, {}    for qi in qids:        pos = pos_map[qi]; gold = TRAIN[qi]["answer"]; q = TRAIN[qi]["question"]        bs, ba = -1.0, None        for row in dict.fromkeys(cand[pos, :k]):            s = meteor_1(gold, assemble([row], q, r))            if s > bs: bs, ba = s, row        best[qi], arg[qi] = bs, ba    return best, argif RUN["S11_headroom"]:    ids = EVAL_IDS[:400]          # oracle đắt -> mẫu nhỏ hơn, vẫn đủ để ra quyết định    r = CFG["R_DEFAULT"]    POS200 = {qi: k for k, qi in enumerate(ids)}      # CAND200 index theo thứ tự `ids`    with T("retrieve top-200 (union rộng, xấp xỉ oracle@∞)"):        CAND200 = cache("cand_train_200",            lambda: np.concatenate([retrieve([TRAIN[i]["question"] for i in ids[s:s+200]],                                             [QROW[i] for i in ids[s:s+200]], topk=200)                                    for s in range(0, len(ids), 200)], 0), "npy")    with T("oracle@200 / oracle@50"):        o200, _ = oracle_at(ids, CAND200, POS200, 200, r)        o50,  _ = oracle_at(ids, CAND200, POS200, 50,  r)    rr1 = {}    for qi in ids:        rows = sorted(RR0[qi].items(), key=lambda kv: -kv[1]) if qi in RR0 else []        top = [rows[0][0]] if rows else [CAND_TR[TR_POS[qi], 0]]        rr1[qi] = meteor_1(TRAIN[qi]["answer"], assemble(top, TRAIN[qi]["question"], r))    O200 = np.mean([o200[i] for i in ids]); O50 = np.mean([o50[i] for i in ids])    R1   = np.mean([rr1[i]  for i in ids])    gaps = {"assembly (1-oracle@200)": 1.0 - O200,            "recall (oracle@200-oracle@50)": O200 - O50,            "precision (oracle@50-reranked@1)": O50 - R1}    print(f"oracle@200 = {O200:.4f}\noracle@50  = {O50:.4f}\nreranked@1 = {R1:.4f}\n")    for k, v in sorted(gaps.items(), key=lambda kv: -kv[1]):        print(f"  {k:<38} {v:.4f}")    winner = max(gaps, key=gaps.get)    print(f"\n>>> BOTTLENECK: {winner}")    print(">>> assembly  -> ưu tiên S07 rule + S18 generator")    print(">>> recall    -> ưu tiên finetune EMBEDDING (đổi thứ tự, bỏ qua S14 trước)")    print(">>> precision -> ưu tiên finetune RERANKER (S13+S14, đúng plan mặc định)")    json.dump({k: float(v) for k, v in gaps.items()} | {"oracle200": float(O200),              "oracle50": float(O50), "reranked1": float(R1)},              open(ART / "headroom.json", "w"), indent=2)

## S12 · Quét `r*` (tỉ lệ độ dài)Công thức dự đoán vùng ngọt `r ≈ 2–2.5`. Cell này xác nhận bằng số thật, và tách theo `qtype`vì câu `"Mẫu ..."` và câu `"...bị xử phạt thế nào?"` có phân phối độ dài rất khác.

In [ ]:
if RUN["S12_rsweep"]:    ids = EVAL_IDS[:600]    def top_rows(qi, n=3):        if qi in RR0:            return [x[0] for x in sorted(RR0[qi].items(), key=lambda kv: -kv[1])[:n]]        return list(CAND_TR[TR_POS[qi], :n])    res = {}    for r in CFG["R_GRID"]:        m = np.array([meteor_1(TRAIN[i]["answer"], assemble(top_rows(i), TRAIN[i]["question"], r))                      for i in ids])        res[r] = m.mean(); print(f"  r={r:<4} METEOR={m.mean():.4f}")    R_STAR = max(res, key=res.get)    print(f"\n>>> r* = {R_STAR}  (mặc định đang là {CFG['R_DEFAULT']})")    print("\nr tốt nhất theo qtype:")    by = {}    for i in ids: by.setdefault(qtype(TRAIN[i]["question"]), []).append(i)    R_BY_TYPE = {}    for t, tids in sorted(by.items(), key=lambda kv: -len(kv[1])):        if len(tids) < 20: continue        best, bs = None, -1        for r in CFG["R_GRID"]:            s = np.mean([meteor_1(TRAIN[i]["answer"], assemble(top_rows(i), TRAIN[i]["question"], r))                         for i in tids])            if s > bs: bs, best = s, r        R_BY_TYPE[t] = best        print(f"  {t:<14} n={len(tids):<5} r*={best:<4} METEOR={bs:.4f}")    R_BY_TYPE["__global__"] = R_STAR    json.dump(R_BY_TYPE, open(ART / "r_by_type.json", "w"))    CFG["R_DEFAULT"] = R_STAR

## S13 · Nhãn METEOR cho reranker — **nhãn trên đúng hàm mục tiêu**Với mỗi cặp `(question, chunk)` ta tính được METEOR **thật** mà chunk đó đạt:```label(q, chunk) = METEOR( assemble([chunk], q, r*), gold(q) )```Tốt hơn silver-citation ở bốn điểm: không nhiễu parse; phủ 7000/7000 (không chỉ 6760); vẫn đúng khi goldtrích văn bản **khác** cái được citation (rất hay xảy ra với bản hợp nhất/sửa đổi); và phân biệt đượcchunk "đúng luật nhưng phủ ít token".`assemble()` dùng ở đây **phải y hệt** lúc inference — đó là toàn bộ lý do S07 là nguồn chân lý duy nhất.

In [ ]:
from multiprocessing import Pooldef _label_one(args):    qi, q, gold, rows, r = args    return qi, [(int(row), float(meteor_1(gold, assemble([row], q, r)))) for row in rows]if RUN["S13_labels"]:    rng = np.random.default_rng(SEED)    lab_ids = list(rng.choice(TRAIN_IDS, size=min(CFG["LABEL_N_TRAIN"], len(TRAIN_IDS)), replace=False))    K = CFG["LABEL_TOP_K"]; r = CFG["R_DEFAULT"]    jobs = [(qi, TRAIN[qi]["question"], TRAIN[qi]["answer"],             list(dict.fromkeys(CAND_TR[TR_POS[qi], :K])), r) for qi in lab_ids]    print(f"{len(jobs)} query × ~{K} chunk = ~{len(jobs)*K:,} lần gọi METEOR")    def _build():        nw = max(1, (os.cpu_count() or 2))        out = {}        with Pool(nw) as p:            for n, (qi, lst) in enumerate(p.imap_unordered(_label_one, jobs, chunksize=16)):                out[qi] = lst                if n % 500 == 0: print(f"   {n}/{len(jobs)}", flush=True)        return out    with T("METEOR labels"):        LABELS = cache("meteor_labels", _build)    v = np.array([s for lst in LABELS.values() for _, s in lst])    mx = np.array([max(s for _, s in lst) for lst in LABELS.values()])    print(f"labels {len(v):,} | mean {v.mean():.4f} | best-per-query mean {mx.mean():.4f}")

## S14 · Finetune reranker — listwise KL trên nhãn METEORListwise (softmax trên nhóm) thay vì pointwise regression: khớp đúng việc reranker phải làm là **xếp hạngtương đối**, và giữ được thông tin có mức độ ("chunk này 0.62, chunk kia 0.58") mà contrastive nhị phânsẽ ném bỏ.Bộ nhớ 16GB T4: group 8, `max_len=512`, gradient checkpointing, AMP fp16.Params fp32 2.27GB + grads 2.27GB + Adam m/v 4.55GB ≈ 9.1GB, activations nhỏ nhờ checkpointing → vừa.`DataParallel` cho 2 GPU (DDP trong notebook rất dễ vỡ; DataParallel kém hiệu quả hơn nhưng đủ cho 568M).

In [ ]:
_HAVE_FT = (FT_DIR / "config.json").exists() and "reranker_ft" not in FORCEif _HAVE_FT:    print(f"[skip] đã có checkpoint tại {FT_DIR} -> KHÔNG train lại.")    print("       (quan trọng với notebook biến thể B/C: chúng dùng cùng reranker,")    print("        không guard chỗ này thì mỗi biến thể tốn lại ~4h GPU vô ích)")    print("       Muốn train lại: FORCE.add('reranker_ft')")if RUN["S14_finetune"] and not _HAVE_FT:    from torch.utils.data import Dataset, DataLoader    import torch.nn.functional as F    from transformers import AutoTokenizer, AutoModelForSequenceClassification    tok_ft = AutoTokenizer.from_pretrained(M_RERANK)    G, T_ = CFG["FT_GROUP"], CFG["FT_TEMP"]    class ListwiseDS(Dataset):        def __init__(s, labels):            s.g = []            for qi, lst in labels.items():                lst = sorted(lst, key=lambda kv: -kv[1])                if len(lst) < G or lst[0][1] <= 0: continue                pos = lst[0]                negs = lst[1:]                pick = [pos] + [negs[i] for i in                        np.random.default_rng(hash(qi) % 2**31).choice(len(negs),                        min(G-1, len(negs)), replace=False)]                if len(pick) < G: continue                s.g.append((TRAIN[qi]["question"], pick))        def __len__(s): return len(s.g)        def __getitem__(s, i):            q, pick = s.g[i]            return q, [CH_TEXT[r] for r, _ in pick], np.array([v for _, v in pick], dtype=np.float32)    def collate(b):        qs, ps, ys = [], [], []        for q, texts, y in b:            qs += [q]*len(texts); ps += texts; ys.append(y)        enc = tok_ft(qs, ps, padding=True, truncation=True,                     max_length=CFG["FT_MAXLEN"], return_tensors="pt")        return enc, torch.tensor(np.stack(ys))    ds = ListwiseDS(LABELS)    print("groups:", len(ds))    dl = DataLoader(ds, batch_size=max(1, len(DEVICES)), shuffle=True,                    collate_fn=collate, num_workers=2, drop_last=True)    mdl = AutoModelForSequenceClassification.from_pretrained(M_RERANK, num_labels=1)    mdl.gradient_checkpointing_enable()    mdl = mdl.to(DEVICES[0])    if len(DEVICES) > 1: mdl = torch.nn.DataParallel(mdl, device_ids=list(range(len(DEVICES))))    opt = torch.optim.AdamW(mdl.parameters(), lr=CFG["FT_LR"], weight_decay=0.01)    scaler = torch.cuda.amp.GradScaler()    steps = CFG["FT_EPOCHS"] * len(dl)    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["FT_LR"], total_steps=steps,                                                pct_start=0.1)    mdl.train(); step = 0    for ep in range(CFG["FT_EPOCHS"]):        for enc, y in dl:            enc = {k: v.to(DEVICES[0]) for k, v in enc.items()}            y = y.to(DEVICES[0])            with torch.autocast("cuda", dtype=AMP_DTYPE):                logits = mdl(**enc).logits.view(y.shape[0], -1)            tgt = torch.softmax(y / T_, dim=-1)            loss = F.kl_div(torch.log_softmax(logits.float(), -1), tgt, reduction="batchmean")            scaler.scale(loss).backward()            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()            step += 1            if step % 25 == 0:                print(f"  ep{ep} step {step}/{steps} loss {loss.item():.4f} "                      f"mem {torch.cuda.max_memory_allocated()/2**30:.1f}GiB", flush=True)    (mdl.module if hasattr(mdl, "module") else mdl).save_pretrained(FT_DIR)    tok_ft.save_pretrained(FT_DIR)    del mdl, opt; gc.collect(); torch.cuda.empty_cache()    print("saved ->", FT_DIR)    # so sánh zero-shot vs finetuned trên cùng tập eval    ids = EVAL_IDS[:400]    RR1 = cache("rr_ft_eval", lambda: rerank_set(ids, TRAIN, CAND_TR, CFG["TOP_FUSE"], ckpt=str(FT_DIR)))    def _m(rr):        return np.mean([meteor_1(TRAIN[i]["answer"],                assemble([max(rr[i], key=rr[i].get)], TRAIN[i]["question"], CFG["R_DEFAULT"]))                for i in ids if i in rr])    print(f"reranked@1 zero-shot {_m(RR0):.4f}  ->  finetuned {_m(RR1):.4f}")

## S15–S16 · Candidate K=6 + Selector (LightGBM, **0 tham số**)Selector chọn cây thay vì head neural **chính vì** luật 4B: nó là thành phần học được mà không tốn mộttham số nào trong ngân sách. Trong cuộc thi bị chặn tham số, dồn trí thông minh vào các khối 0-param(BM25, regex citation, rule assembly, LightGBM) là arbitrage.Selector làm **hai** việc: chọn candidate, **và** quyết định khi nào bật floor guard ở S17.

In [ ]:
CAND_SPECS = [    dict(name="top1_r1.0", n=1, r=1.0, expand=True),    dict(name="top1_rstar", n=1, r=None, expand=True),    dict(name="top1_rhigh", n=1, r=None, expand=True, rmul=1.4),    dict(name="top2_rstar", n=2, r=None, expand=True),    dict(name="top3_rstar", n=3, r=None, expand=True),    dict(name="top1_noexp", n=1, r=None, expand=False),]def make_candidates(question, ranked_rows, scores):    out = []    rstar = CFG["R_DEFAULT"]    for sp in CAND_SPECS:        r = (sp["r"] or rstar) * sp.get("rmul", 1.0)        txt = assemble(ranked_rows[:sp["n"]], question, r, expand=sp["expand"])        out.append((sp["name"], txt, r, sp["n"]))    return outdef feats(question, txt, r, n, ranked_rows, scores):    cq = parse_cit(question); ct = parse_cit(txt)    s = sorted(scores, reverse=True) + [0.0, 0.0]    w = len(txt.split())    tgt = target_words(question, r)    toks = txt.split()    uniq = len(set(toks)) / max(1, len(toks))    return [s[0], s[1], s[0]-s[1], float(np.mean(s[:5])), r, n,            w, w/max(1,tgt), len(question.split()),            float(bool(cq["van_ban"])), float(bool(cq["dieu"])),            float(len(set(cq["van_ban"]) & set(ct["van_ban"])) > 0),            float(len(set(cq["dieu"]) & set(ct["dieu"])) > 0),            len(ct["van_ban"]), len(ct["dieu"]), uniq,            float(txt.count("\n")), float(len(re.findall(r"\d", txt))/max(1,w)),    ] + [1.0 if qtype(question) == t else 0.0 for t, _ in QTYPE_RULES]FEAT_NAMES = ["s1","s2","margin","s_mean5","r","n_span","words","words_over_tgt","q_words",              "q_has_vb","q_has_dieu","vb_match","dieu_match","n_vb","n_dieu","uniq_ratio",              "n_lines","digit_ratio"] + [f"qt_{t}" for t, _ in QTYPE_RULES]if RUN["S15_candidates"]:    RR_USE = None    if (FT_DIR / "config.json").exists():        ids_all = EVAL_IDS        RR_USE = cache("rr_ft_sel", lambda: rerank_set(ids_all, TRAIN, CAND_TR,                                                       CFG["TOP_FUSE"], ckpt=str(FT_DIR)))    else:        RR_USE = RR0        print("[warn] chưa có reranker finetune -> dùng zero-shot cho selector")    def _build_sel():        X, Y, Q, S = [], [], [], []        for qi in [i for i in EVAL_IDS if i in RR_USE]:            rr = sorted(RR_USE[qi].items(), key=lambda kv: -kv[1])            rows = [a for a, _ in rr]; scs = [b for _, b in rr]            q, gold = TRAIN[qi]["question"], TRAIN[qi]["answer"]            for nm, txt, r, n in make_candidates(q, rows, scs):                if not txt: continue                X.append(feats(q, txt, r, n, rows, scs))                Y.append(meteor_1(gold, txt)); Q.append(qi); S.append(nm)        return dict(X=np.array(X, dtype=np.float32), Y=np.array(Y, dtype=np.float32),                    Q=np.array(Q), S=np.array(S))    with T("selector dataset"):        SEL = cache("selector_data", _build_sel)    print("X", SEL["X"].shape)if RUN["S16_selector"]:    groups = np.array([QGROUP[q] for q in SEL["Q"]])    from sklearn.model_selection import GroupKFold    try:        import lightgbm as lgb        HAS_LGB = True    except ImportError:        from sklearn.ensemble import HistGradientBoostingRegressor        HAS_LGB = False    oof = np.zeros(len(SEL["Y"]), dtype=np.float32); models = []    for tr, te in GroupKFold(n_splits=5).split(SEL["X"], groups=groups):        if HAS_LGB:            m = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31,                                  min_child_samples=30, subsample=0.9, colsample_bytree=0.9,                                  random_state=SEED, verbose=-1)        else:            m = HistGradientBoostingRegressor(max_iter=400, random_state=SEED)        m.fit(SEL["X"][tr], SEL["Y"][tr]); oof[te] = m.predict(SEL["X"][te]); models.append(m)    print("OOF corr(pred, METEOR):", np.corrcoef(oof, SEL["Y"])[0,1].round(4))    # so sánh: selector vs luôn chọn 1 spec cố định (group theo TÊN spec, không theo stride —    # candidate rỗng bị skip nên stride sai)    dfq = pd.DataFrame(dict(q=SEL["Q"], y=SEL["Y"], p=oof, spec=SEL["S"]))    picked = dfq.loc[dfq.groupby("q")["p"].idxmax(), "y"].mean()    fixed = dfq.groupby("spec")["y"].mean().sort_values(ascending=False)    oracle = dfq.groupby("q")["y"].max().mean()    print(fixed.round(4).to_string())    print(f"\nspec cố định tốt nhất {fixed.iloc[0]:.4f} | selector {picked:.4f} "          f"| oracle candidate {oracle:.4f}")    if picked <= fixed.iloc[0]:        print("[warn] selector KHÔNG thắng spec cố định -> dùng spec cố định, đừng thêm phức tạp vô ích")    pickle.dump(models, open(ART / "selector.pkl", "wb"))    SEL_MODELS = models

## S17 · Inference public + floor guard + validator + submissionFloor guard: điểm mấu chốt là METEOR final là **trung bình 1000 câu**, nên tiền nằm ở **đuôi dưới**(kéo 100 câu từ 0.10→0.35 bằng đúng kéo 100 câu từ 0.55→0.80, nhưng dễ hơn nhiều lần). Vì vậy:không bao giờ xuất chuỗi ngắn / rỗng / `"None"` / lời xin lỗi — kém nhất cũng phải là scaffold đúng độ dài.Validator chặn cứng theo `scoring.py`: thiếu/thừa key ⇒ `KeyError` ⇒ **chết toàn bộ bài nộp**.

In [ ]:
def predict_public():    if CFG["SPARSE_ONLY"]:        # CPU-only: không có reranker -> dùng thứ tự RRF của BM25 (+citation boost)        print("[SPARSE_ONLY] bỏ reranker, dùng thứ hạng BM25")        rr = {qi: {int(r): float(CFG["TOP_FUSE"] - k)                   for k, r in enumerate(dict.fromkeys(CAND_PB[PB_POS[qi]]))}              for qi in PUBLIC_IDS}    else:        rr = cache("rr_public", lambda: rerank_set(            PUBLIC_IDS, PUBLIC, CAND_PB, CFG["TOP_FUSE"],            ckpt=str(FT_DIR) if (FT_DIR/"config.json").exists() else None))    out, n_dup, n_floor = {}, 0, 0    for qi in PUBLIC_IDS:        if qi in DUP_GATE:                                   # gate trùng train            out[qi] = TRAIN[DUP_GATE[qi]]["answer"]; n_dup += 1; continue        q = PUBLIC[qi]["question"]        srt = sorted(rr[qi].items(), key=lambda kv: -kv[1])        rows = [a for a, _ in srt]; scs = [b for _, b in srt]        cands = [(nm, t, r, n) for nm, t, r, n in make_candidates(q, rows, scs) if t]        if cands and 'SEL_MODELS' in globals():            Xc = np.array([feats(q, t, r, n, rows, scs) for _, t, r, n in cands], dtype=np.float32)            pred = np.mean([m.predict(Xc) for m in SEL_MODELS], 0)            k = int(np.argmax(pred)); txt, best_pred = cands[k][1], float(pred[k])        elif cands:            # fallback: spec 'top1_rstar' nếu có, không thì candidate đầu tiên            pick = next((c for c in cands if c[0] == "top1_rstar"), cands[0])            txt, best_pred = pick[1], 1.0        else:            txt, best_pred = "", -1.0        if (not txt) or len(txt.split()) < 40 or best_pred < 0.03:   # floor guard            tgt = target_words(q, CFG["R_DEFAULT"])            base = assemble(rows[:3], q, CFG["R_DEFAULT"]) or ""            if len(base.split()) < 40:                reps = max(1, math.ceil(tgt / len(SCAFFOLD.split())))                base = _trim("\n".join([SCAFFOLD]*reps), tgt)            txt = base; n_floor += 1        out[qi] = txt    print(f"dup gate {n_dup} | floor guard {n_floor}")    return outdef validate_and_pack(pred: dict, tag="run"):    assert set(pred) == set(PUBLIC_IDS), \        f"key lệch: thiếu {len(set(PUBLIC_IDS)-set(pred))}, thừa {len(set(pred)-set(PUBLIC_IDS))}"    sub = {}    for k, v in pred.items():        assert isinstance(v, str) and v.strip() and v.strip().lower() != "none", f"answer xấu tại {k}"        sub[k] = {"answer": v}    # Tên file NGOÀI có tag, tên file TRONG zip luôn là submission.json (Codabench yêu cầu).    # Không tag tên zip thì chạy A rồi B/C sẽ ghi đè lẫn nhau và không biết zip nào của đâu.    p = SUB / f"submission_{tag}.json"    with io.open(p, "w", encoding="utf-8") as f:        json.dump(sub, f, ensure_ascii=False)    import zipfile    z = SUB / f"submission_{tag}.zip"    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as zf:        zf.write(p, arcname="submission.json")       # submission.json ở ROOT zip    sha = hashlib.sha256(open(p, "rb").read()).hexdigest()    w = np.array([len(v["answer"].split()) for v in sub.values()])    man = dict(tag=tag, n=len(sub), sha256_json=sha,               words=dict(mean=float(w.mean()), median=float(np.median(w)),                          p10=float(np.percentile(w,10)), max=int(w.max())),               models=dict(emb=M_EMB, rerank=M_RERANK,                           rerank_ft=str(FT_DIR) if (FT_DIR/"config.json").exists() else None,                           gen=M_GEN if RUN["S18_generator"] else None),               cfg={k: v for k, v in CFG.items() if not isinstance(v, (dict,))},               seed=SEED, headroom=json.load(open(ART/"headroom.json"))                          if (ART/"headroom.json").exists() else None)    json.dump(man, open(SUB / f"manifest_{tag}.json", "w"), indent=2, ensure_ascii=False)    print(json.dumps(man["words"], indent=2))    print("sha256:", sha, "\nzip:", z, f"({z.stat().st_size/2**20:.1f} MiB)")    return zdef list_submissions():    zs = sorted(SUB.glob("submission_*.zip"))    print(f"\n{'file':<34}{'MiB':>8}{'n':>7}  sha256[:12]")    for z in zs:        man = SUB / f"manifest_{z.stem.replace('submission_','')}.json"        m = json.load(open(man)) if man.exists() else {}        print(f"{z.name:<34}{z.stat().st_size/2**20:>8.2f}{m.get('n','?'):>7}  "              f"{str(m.get('sha256_json',''))[:12]}")    print("\nTải về: tab Output của notebook Kaggle -> submit/<file>.zip")    return zsif RUN["S17_submit"]:    with T("inference public"):        PRED = predict_public()    Z = validate_and_pack(PRED, tag="v1")    k0 = PUBLIC_IDS[0]    print("\n--- ví dụ ---\nQ:", PUBLIC[k0]["question"][:180])    print("A:", PRED[k0][:700].replace("\n", " | "))    list_submissions()

## S18 · *(optional)* Generator grounded SFT**Chỉ bật khi S11 cho thấy `assembly` là khoảng trống lớn nhất.** Lý do nó ở cuối là **phụ thuộc**, khôngchỉ ưu tiên: context train lấy từ retriever, nên retriever đổi ⇒ phân phối context đổi ⇒ checkpoint SFT lệch.Và **chỉ dạng grounded**: `question + context → answer`. Dạng `question → answer` (không context) dạy modelsinh nội dung luật **từ trí nhớ** — nó sẽ học văn phong hoàn hảo và bịa số Điều, mức phạt, con số. Vừa vi phạmyêu cầu "trích dẫn căn cứ" của thể lệ, vừa vỡ ở Private Test.Vì gold phần lớn là nguyên văn từ context, teacher forcing dạy đúng thứ ta cần: **copy span, giữ nguyên sốvà dấu câu, dừng đúng chỗ**. LoRA chỉ để tiết kiệm VRAM — theo luật BTC nó **không** làm giảm số tham số.

In [ ]:
if RUN["S18_generator"]:    try:        import peft    except ImportError:        pip("peft"); import peft    from peft import LoraConfig, get_peft_model    from transformers import AutoModelForCausalLM    SYS = ("Bạn là trợ lý pháp luật. CHỈ được sao chép nguyên văn các quy định từ NGỮ CẢNH. "           "Không diễn giải, không tóm tắt, không thêm thông tin ngoài ngữ cảnh. "           "Giữ nguyên số, đơn vị, dấu câu, cách đánh số và gạch đầu dòng.")    def build_prompt(question, rows, max_ctx_words=1200):        ctx, n = [], 0        for ri in rows[:3]:            blk = _head(ri) + "\n" + _full_dieu(ri).strip()            w = len(blk.split())            if n + w > max_ctx_words:                blk = _trim(blk, max_ctx_words - n); w = len(blk.split())            ctx.append(blk); n += w            if n >= max_ctx_words: break        return (f"<|im_start|>system\n{SYS}<|im_end|>\n<|im_start|>user\n"                f"NGỮ CẢNH:\n{chr(10).join(ctx)}\n\nCÂU HỎI: {question}\n\n"                f"Trả lời bằng cách trích nguyên văn căn cứ pháp lý.<|im_end|>\n<|im_start|>assistant\n")    tok_g = AutoTokenizer.from_pretrained(M_GEN)    if tok_g.pad_token is None: tok_g.pad_token = tok_g.eos_token    gen = AutoModelForCausalLM.from_pretrained(M_GEN, torch_dtype=AMP_DTYPE).to(DEVICES[0])    gen.gradient_checkpointing_enable(); gen.enable_input_require_grads()    gen = get_peft_model(gen, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],        task_type="CAUSAL_LM"))    gen.print_trainable_parameters()    # ~10% mẫu có context CỐ TÌNH SAI -> dạy model không tin context mù quáng    rng = np.random.default_rng(SEED)    sft_ids = [i for i in list(rng.choice(EVAL_IDS, min(3000, len(EVAL_IDS)), replace=False))               if i in RR_USE]    ex = []    for qi in sft_ids:        rows = [a for a, _ in sorted(RR_USE[qi].items(), key=lambda kv: -kv[1])]        if rng.random() < 0.10:            rows = list(rng.choice(len(CH), 3, replace=False))        ex.append((build_prompt(TRAIN[qi]["question"], rows), TRAIN[qi]["answer"]))    print("SFT examples:", len(ex))    print("\n>>> Vòng huấn luyện SFT để trống có chủ ý: chỉ viết sau khi S11 xác nhận")    print(">>> assembly là bottleneck. Nếu không, BỎ generator -> hệ thống còn 1.14B tham số")    print(">>> và theo phân tích metric thì đó là kết cục hoàn toàn chấp nhận được.")

## S19 · Checklist trước khi nộp- [ ] `submission.zip` chứa **đúng 1 file** `submission.json` ở **gốc zip**- [ ] Số key = **1.000**, trùng khớp hoàn toàn với `public-official.json`- [ ] Mỗi value là object có `answer` là **string không rỗng**, không `null`, không `"None"`- [ ] UTF-8, JSON hợp lệ- [ ] `param_budget()` đã chạy và **tổng < 4.000.000.000** (số đếm thật, không ước lượng)- [ ] Cả 3 model đều nằm trong danh sách BTC duyệt (đối chiếu **Google Sheet**, mới hơn PDF)- [ ] Ghi log: `manifest_*.json` (SHA-256, config, seed, headroom) + phiên bản `nltk`/`rouge_score`- [ ] Nộp bằng **đúng Organization** đã đăng ký trên Codabench**Kỷ luật lượt nộp.** Public 10 lượt/ngày — chỉ nộp khi OOF METEOR tăng ≥ 0.005, và ghi lại`(config, OOF, LB)` mỗi lần để dựng hồi quy offline→online. Private chỉ **3 lượt/ngày trong 5 ngày**và không có leaderboard để dò: ai chưa tin được OOF ở thời điểm đó thì coi như nộp mù.**Còn treo (nên email BTC tuần này):**1. Dùng label Task 1 để train retriever Task 2 có hợp lệ không? (22 câu hỏi trùng giữa 2 train set)2. LightGBM / mô hình ML cổ điển có phải đăng ký như "mô hình" không? (tôi cho là không, nhưng vẫn khai trong Model Card)